In [112]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1/spm.model
/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1/config.json
/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1/tf_model.h5
/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1/tokenizer_config.json
/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1/pytorch_model.bin
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [113]:
from transformers import pipeline,AutoTokenizer,AutoModelForSequenceClassification

from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")

In [114]:
torch.manual_seed(42)
np.random.seed(42)

In [115]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [139]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [116]:
deb_tokenizer=AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
deb=AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-small",num_labels=5)

rob_tokenizer=AutoTokenizer.from_pretrained("roberta-base")
rob=AutoModelForSequenceClassification.from_pretrained("roberta-base",num_labels=5)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [117]:
train['label']=LabelEncoder().fit_transform(train['answer'])
train.head()

,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


In [118]:
label={
    0:'A',
    1:'B',
    2:'C',
    3:'D',
    4:'E'
}

# Question 1

In [119]:
prompt_25=train['prompt'][25]
prompt_25

"Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully."

In [120]:
deb.eval()

deb_input=deb_tokenizer(prompt_25,return_tensors="pt")

with torch.no_grad():
    deb_log=deb(**deb_input).logits
deb_log

tensor([[ 0.1106, -0.1007, -0.0121, -0.0373, -0.3591]], dtype=torch.float16)

In [121]:
deb_probs=F.softmax(deb_log, dim=-1).squeeze()

print("DeBERTa Softmax Probabilities")
for i in (np.argsort(-deb_probs)):
    print(f"{label[int(i)]}: {float(deb_probs[int(i)]):.4f}")

DeBERTa Softmax Probabilities
A: 0.2391
C: 0.2115
D: 0.2063
B: 0.1936
E: 0.1495


In [123]:
rob.eval()

rob_input=rob_tokenizer(prompt_25,return_tensors="pt")

with torch.no_grad():
    rob_log=rob(**rob_input).logits
rob_log

tensor([[-0.0876, -0.0398, -0.0474,  0.0293, -0.0966]])

In [124]:
rob_probs=F.softmax(rob_log, dim=-1).squeeze()

print("RoBERTa Softmax Probabilities")
for i in (np.argsort(-rob_probs)):
    print(f"{label[int(i)]}: {float(rob_probs[int(i)]):.4f}")

RoBERTa Softmax Probabilities
D: 0.2159
B: 0.2015
C: 0.2000
A: 0.1921
E: 0.1904


# Question 2

In [128]:
print("Average Softmax Probabilities")
avg_probs=(deb_probs+rob_probs)/2
for i in (np.argsort(-avg_probs)):
    print(f"{label[int(i)]}: {float(avg_probs[int(i)]):.4f}")


Average Softmax Probabilities
A: 0.2156
D: 0.2111
C: 0.2058
B: 0.1976
E: 0.1700


# Question 3

In [129]:
print("Weighted Average Softmax Probabilities")
weigthed_avg_probs=(0.7*deb_probs+0.3*rob_probs)
for i in (np.argsort(-weigthed_avg_probs)):
    print(f"{label[int(i)]}: {float(weigthed_avg_probs[int(i)]):.4f}")


Weighted Average Softmax Probabilities
A: 0.2250
D: 0.2092
C: 0.2081
B: 0.1960
E: 0.1618


# Question 4

In [138]:
print("Top 3 Choices")
print(' '.join(label[int(i)] for i in (np.argsort(-weigthed_avg_probs)[:3])))

Top 3 Choices
A D C


# Question 5

In [141]:
preds=[]
deb.eval()
rob.eval()
for _,row in test.iterrows():    
    
    deb_input=deb_tokenizer(row['prompt'],return_tensors="pt")
    rob_input=rob_tokenizer(row['prompt'],return_tensors="pt")
    
    with torch.no_grad():
        deb_log=deb(**deb_input).logits
        rob_log=rob(**rob_input).logits

    deb_probs=F.softmax(deb_log, dim=-1).squeeze()
    rob_probs=F.softmax(rob_log, dim=-1).squeeze()

    weigthed_avg_probs=(0.7*deb_probs+0.3*rob_probs)

    preds.append(' '.join(label[int(i)] for i in (np.argsort(-weigthed_avg_probs)[:3])))
preds[:5] 

['B C A', 'C B A', 'A B C', 'C A B', 'C D B']

In [142]:
sub=pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sub.set_index('ID',inplace=True)
sub.head()

,Prediction
ID,
1,A B C
2,A B C
3,A B C
4,A B C
5,A B C


In [149]:
sub.Prediction=preds
sub.head()

,Prediction
ID,
1,B C A
2,C B A
3,A B C
4,C A B
5,C D B


In [150]:
sub.to_csv("submission.csv")

In [148]:
print("Number of Predicted Rows:",sub.shape[0])

Number of Predicted Rows: 500


# Question 6